In [3]:
import numpy as np
import pandas as pd

In [4]:
df= pd.read_csv("test_data.csv")

game_map= np.zeros((6,6))

for _, row in df.iterrows():
    game_map[row['cell_index']//6][row['cell_index']%6] = row['type']

In [5]:
def in_matrix(row, col):
    return row>=0 and row<=5 and col>=0 and col<=5

row_dir=[-1, 0, 1, 0] 
col_dir=[0, 1, 0, -1] # up right down left

def step(row, col, action):
    new_row, new_col= row+ row_dir[action], col+ col_dir[action]
    if not in_matrix(new_row, new_col):
        return row, col
    return new_row, new_col


In [6]:
gamma= 0.9
step_penalty=-0.01
win= 1.0

V= np.zeros((6,6))

V[5, 5]= 0

threshold= 1e-6

while True:
    delta= 0.0
    new_V= V.copy()
    for row in range(6):
        for col in range(6):
            if game_map[row][col] in [1, 3]:
                continue
            
            best_reward= -np.inf
            for i in range(4):
                reward= 0.0

                for prob, action in [(0.8, i), (0.1, (i-1)% 4), (0.1, (i+1)% 4)]:
                    new_row, new_col= step(row, col, action)

                    reward+= ((win if game_map[new_row][new_col]== 3 else step_penalty)+ gamma * V[new_row][new_col]) * prob
                best_reward= max(reward, best_reward)
            
            new_V[row][col]= best_reward
            delta= max(delta, abs(new_V[row][col]- V[row][col]))
    
    V= new_V

    if delta< threshold:
        break


In [7]:
best_actions= np.zeros((6, 6))

for row in range(6):
    for col in range(6):
        if game_map[row][col] in [1, 3]:
            continue
        best_action= 0
        best_value= -np.inf
        for i in range(4):
            reward= 0.0

            for prob, action in [(0.8, i), (0.1, (i-1)% 4), (0.1, (i+1)% 4)]:
                new_row, new_col= step(row, col, action)

                if game_map[new_row][new_col] == 3:
                    r = win
                else:
                    r = step_penalty

                reward += (r + gamma * V[new_row][new_col]) * prob

            if reward > best_value:
                best_value= reward
                best_action= i
        best_actions[row][col]= best_action

In [8]:
answer= []

for row in range(6):
    for col in range(6):
        answer.append({
            "subtaskID":1,
            "datapointID": row*6+ col,
            "answer": V[row][col]
        })
        if game_map[row][col] in [1, 3]:
            continue
        answer.append({
            "subtaskID":2,
            "datapointID": row*6+ col,
            "answer": best_actions[row][col]
        })

pd.DataFrame(answer).to_csv("submission.csv", index= False)